In [ ]:
import pandas as pd
import geopandas as gpd
import os, re
from shapely.geometry import Point, Polygon, MultiPolygon
import numpy as np

import numpy as np
from typing import Literal

from datetime import datetime, time
import geopandas as gpd
import xarray as xr

import matplotlib.pyplot as plt


import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import rioxarray

import requests
import json
from shapely.geometry import shape


In [ ]:
crs_standard = 'EPSG:4326'

In [ ]:
file_path_aoi = "/Users/ryanmc/Documents/Complex_Risk_Science/dev/Complex-Risk-Collective/.github/Projects/NASA-disasters-grid-resilience/data/ca_and_bordering_states.geojson"
gdf_aoi = gpd.read_file(file_path_aoi)
gdf_aoi = gdf_aoi.to_crs(crs_standard)

In [ ]:
gdf_aoi = gdf_aoi[gdf_aoi['name']=='CALIFORNIA']
gdf_aoi

### HIFLD

In [ ]:
#HIFLD 

file_path_powergrid = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/U.S._Electric_Power_Transmission_Lines.geojson"
gdf_powergrid_HIFLD = gpd.read_file(file_path_powergrid)

# Convert timeframes to folium-friendly types
gdf_powergrid_HIFLD['SOURCEDATE'] = pd.to_datetime(gdf_powergrid_HIFLD['SOURCEDATE']).dt.strftime('%Y-%m-%dT%H:%M:%S')
gdf_powergrid_HIFLD['VAL_DATE'] = pd.to_datetime(gdf_powergrid_HIFLD['VAL_DATE']).dt.strftime('%Y-%m-%dT%H:%M:%S')


# Drop columns with 'NOT AVAILABLE' as substation value
# gdf[ (gdf['SUB_1']=='NOT AVAILABLE') | (gdf['SUB_2']=='NOT AVAILABLE')]
condition = ( (gdf_powergrid_HIFLD['SUB_1']=='NOT AVAILABLE') | (gdf_powergrid_HIFLD['SUB_2']=='NOT AVAILABLE') )
gdf_powergrid_HIFLD = gdf_powergrid_HIFLD[~condition]
# gdf

# Drop columns with 'NONE' as substation value
condition = ( (gdf_powergrid_HIFLD['SUB_1']=='NONE') | (gdf_powergrid_HIFLD['SUB_2']=='NONE') )
gdf_powergrid_HIFLD = gdf_powergrid_HIFLD[~condition]
gdf_powergrid_HIFLD



In [ ]:
# gdf_powergrid_HIFLD.iloc[10000]#['VAL_DATE']


# Distribution plots for HIFLD power grid source data

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('HIFLD Transmission Lines — Source Data Distributions', fontsize=14, fontweight='bold')

# ── 1. SOURCEDATE distribution ──────────────────────────────────────────────
ax = axes[0, 0]
source_years = pd.to_datetime(gdf_powergrid_HIFLD['SOURCEDATE']).dt.year.dropna()
source_years.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_title('Source Date Distribution (by Year)', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# ── 2. VAL_DATE distribution ─────────────────────────────────────────────────
ax = axes[0, 1]
val_years = pd.to_datetime(gdf_powergrid_HIFLD['VAL_DATE']).dt.year.dropna()
val_years.value_counts().sort_index().plot(kind='bar', ax=ax, color='darkorange', edgecolor='white', linewidth=0.5)
ax.set_title('Validation Date Distribution (by Year)', fontsize=11)
ax.set_xlabel('Year')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# ── 3. SOURCE field distribution ─────────────────────────────────────────────
ax = axes[1, 0]
source_counts = gdf_powergrid_HIFLD['SOURCE'].value_counts()
top_n = 15  # limit to top N sources for readability
source_counts.head(top_n).plot(kind='barh', ax=ax, color='seagreen', edgecolor='white', linewidth=0.5)
ax.set_title(f'Top {top_n} Data Sources (SOURCE field)', fontsize=11)
ax.set_xlabel('Count')
ax.set_ylabel('Source')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3, linestyle='--')

# ── 4. SOURCEDATE vs VAL_DATE scatter (temporal alignment) ──────────────────
ax = axes[1, 1]
sd = pd.to_datetime(gdf_powergrid_HIFLD['SOURCEDATE'])
vd = pd.to_datetime(gdf_powergrid_HIFLD['VAL_DATE'])
ax.scatter(sd, vd, alpha=0.15, s=5, color='mediumpurple')
ax.plot(
    [sd.min(), sd.max()], [sd.min(), sd.max()],
    color='red', linestyle='--', linewidth=1, label='1:1 line'
)
ax.set_title('Source Date vs Validation Date', fontsize=11)
ax.set_xlabel('Source Date')
ax.set_ylabel('Validation Date')
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=30)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()

# ── Summary stats ────────────────────────────────────────────────────────────
print("=== SOURCE field summary ===")
print(f"Unique sources: {gdf_powergrid_HIFLD['SOURCE'].nunique()}")
print(f"Top 5 sources:\n{gdf_powergrid_HIFLD['SOURCE'].value_counts().head(5).to_string()}")
print(f"\nSOURCEDATE range: {source_years.min()} – {source_years.max()}")
print(f"VAL_DATE range:   {val_years.min()} – {val_years.max()}")

In [ ]:
# Clip to ensure only grid elements in the AOI are kept
gdf_powergrid_HIFLD_aoi = gpd.sjoin(gdf_powergrid_HIFLD, gdf_aoi, how="inner", predicate='within').drop(columns=['index_right'])

### CEC

In [ ]:
#CEC

file_path_powergrid_CEC = "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/Transmission_Line_data_CA_CEC/TransmissionLine_CEC.shp"
gdf_powergrid_CEC = gpd.read_file(file_path_powergrid_CEC)




In [ ]:
gdf_powergrid_CEC

In [ ]:
# Side-by-side comparison: CEC vs HIFLD transmission lines over AOI boundary

# Ensure all layers share CRS
gdf_aoi_plot = gdf_aoi.to_crs(crs_standard).copy()
gdf_powergrid_CEC_plot = gdf_powergrid_CEC.to_crs(crs_standard).copy()
gdf_powergrid_HIFLD_plot = gdf_powergrid_HIFLD.to_crs(crs_standard).copy()

# Clip both grid datasets to AOI for fair visual comparison
gdf_powergrid_CEC_aoi = gpd.sjoin(
    gdf_powergrid_CEC_plot, gdf_aoi_plot, how='inner', predicate='within'
).drop(columns=['index_right'])

gdf_powergrid_HIFLD_aoi = gpd.sjoin(
    gdf_powergrid_HIFLD_plot, gdf_aoi_plot, how='inner', predicate='within'
).drop(columns=['index_right'])

# Common extent from AOI
xmin, ymin, xmax, ymax = gdf_aoi_plot.total_bounds

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharex=True, sharey=True)

# Left panel: CEC
gdf_aoi_plot.boundary.plot(ax=axes[0], color='black', linewidth=1.5, zorder=3)
gdf_powergrid_CEC_aoi.plot(ax=axes[0], color='tab:orange', linewidth=0.7, alpha=0.9, zorder=2)
axes[0].set_title(f'CEC Transmission Lines (n={len(gdf_powergrid_CEC_aoi)})', fontsize=12)
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].grid(True, alpha=0.25, linestyle='--')

# Right panel: HIFLD
gdf_aoi_plot.boundary.plot(ax=axes[1], color='black', linewidth=1.5, zorder=3)
gdf_powergrid_HIFLD_aoi.plot(ax=axes[1], color='tab:blue', linewidth=0.7, alpha=0.9, zorder=2)
axes[1].set_title(f'HIFLD Transmission Lines (n={len(gdf_powergrid_HIFLD_aoi)})', fontsize=12)
axes[1].set_xlabel('Longitude')
axes[1].grid(True, alpha=0.25, linestyle='--')

for ax in axes:
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

fig.suptitle('Power Grid Representation Comparison over AOI: CEC vs HIFLD', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"CEC lines in AOI: {len(gdf_powergrid_CEC_aoi)}")
print(f"HIFLD lines in AOI: {len(gdf_powergrid_HIFLD_aoi)}")

In [ ]:
# Delta plot: where CEC and HIFLD line networks agree vs differ

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# Requires gdf_powergrid_CEC_aoi, gdf_powergrid_HIFLD_aoi, and gdf_aoi_plot from previous cells
if 'gdf_powergrid_CEC_aoi' not in globals() or 'gdf_powergrid_HIFLD_aoi' not in globals():
    raise ValueError('Run the prior comparison cell first so AOI-clipped CEC/HIFLD GeoDataFrames are available.')

# Work in projected CRS for distance-based buffering (meters)
delta_crs = 'EPSG:3310'  # California Albers
buffer_m = 1000  # matching tolerance corridor around lines (adjust as needed)

cec_m = gdf_powergrid_CEC_aoi.to_crs(delta_crs).copy()
hifld_m = gdf_powergrid_HIFLD_aoi.to_crs(delta_crs).copy()
aoi_m = gdf_aoi_plot.to_crs(delta_crs).copy()

# Build buffered coverage polygons for robust geometric comparison
cec_cov = cec_m.geometry.buffer(buffer_m).union_all()
hifld_cov = hifld_m.geometry.buffer(buffer_m).union_all()

overlap_cov = cec_cov.intersection(hifld_cov)
cec_only_cov = cec_cov.difference(hifld_cov)
hifld_only_cov = hifld_cov.difference(cec_cov)

# Convert to GeoDataFrames for plotting
gdf_overlap = gpd.GeoDataFrame({'label': ['Overlap']}, geometry=[overlap_cov], crs=delta_crs)
gdf_cec_only = gpd.GeoDataFrame({'label': ['CEC only']}, geometry=[cec_only_cov], crs=delta_crs)
gdf_hifld_only = gpd.GeoDataFrame({'label': ['HIFLD only']}, geometry=[hifld_only_cov], crs=delta_crs)

# Plot in EPSG:4326 for readability
gdf_overlap = gdf_overlap.to_crs('EPSG:4326')
gdf_cec_only = gdf_cec_only.to_crs('EPSG:4326')
gdf_hifld_only = gdf_hifld_only.to_crs('EPSG:4326')
cec_plot = cec_m.to_crs('EPSG:4326')
hifld_plot = hifld_m.to_crs('EPSG:4326')
aoi_plot = aoi_m.to_crs('EPSG:4326')

fig, ax = plt.subplots(1, 1, figsize=(12, 10))

# Base context
aoi_plot.boundary.plot(ax=ax, color='black', linewidth=1.3, zorder=5)
cec_plot.plot(ax=ax, color='darkorange', linewidth=0.3, alpha=0.25, zorder=2)
hifld_plot.plot(ax=ax, color='royalblue', linewidth=0.3, alpha=0.25, zorder=2)

# Delta layers
gdf_overlap.plot(ax=ax, color='seagreen', alpha=0.45, edgecolor='none', zorder=3)
gdf_cec_only.plot(ax=ax, color='orange', alpha=0.55, edgecolor='none', zorder=4)
gdf_hifld_only.plot(ax=ax, color='dodgerblue', alpha=0.55, edgecolor='none', zorder=4)

xmin, ymin, xmax, ymax = aoi_plot.total_bounds
ax.set_xlim(xmin, xmax)
ax.set_ylim(ymin, ymax)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Delta Plot of Transmission-Line Representations (buffer = {buffer_m} m)', fontsize=13, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.25)

# Explicit legend handles (reliable with GeoPandas plots)
legend_handles = [
    Line2D([0], [0], color='black', lw=1.3, label='AOI boundary'),
    Line2D([0], [0], color='darkorange', lw=1.2, alpha=0.7, label='CEC lines (context)'),
    Line2D([0], [0], color='royalblue', lw=1.2, alpha=0.7, label='HIFLD lines (context)'),
    Patch(facecolor='seagreen', edgecolor='none', alpha=0.45, label='Overlap zone'),
    Patch(facecolor='orange', edgecolor='none', alpha=0.55, label='CEC-only zone'),
    Patch(facecolor='dodgerblue', edgecolor='none', alpha=0.55, label='HIFLD-only zone'),
]
ax.legend(handles=legend_handles, loc='lower left', frameon=True)

plt.tight_layout()
plt.show()

# Optional summary metrics (km^2 of corridor zones)
overlap_km2 = gdf_overlap.to_crs(delta_crs).geometry.area.sum() / 1e6
cec_only_km2 = gdf_cec_only.to_crs(delta_crs).geometry.area.sum() / 1e6
hifld_only_km2 = gdf_hifld_only.to_crs(delta_crs).geometry.area.sum() / 1e6

print(f'Overlap corridor area (km^2): {overlap_km2:,.2f}')
print(f'CEC-only corridor area (km^2): {cec_only_km2:,.2f}')
print(f'HIFLD-only corridor area (km^2): {hifld_only_km2:,.2f}')

### Generation system

In [ ]:
file_egrid = '/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/physical_grid_data/eGrid/2023/egrid2023_data_rev2.xlsx'



In [ ]:
import plotly.express as px

# --- Load PLNT23 ---
plnt23 = pd.read_excel(file_egrid, sheet_name="PLNT23")

def pick_col(df, candidates, label):
    c = next((x for x in candidates if x in df.columns), None)
    if c is None:
        raise ValueError(f"Could not find {label}. Available columns sample: {list(df.columns)[:30]}")
    return c

# Common eGRID column names (with fallbacks)
lat_col = pick_col(plnt23, ["LAT", "Plant latitude", "Latitude", "latitude"], "latitude column")
lon_col = pick_col(plnt23, ["LON", "LONG", "Longitude", "longitude", "Plant longitude"], "longitude column")
plant_col = pick_col(plnt23, ["PNAME", "PLANT_NAME", "Plant name"], "plant name column")
cap_col = pick_col(plnt23, ["NAMEPCAP", "NAMEPLATE_CAPACITY_MW", "Plant nameplate capacity (MW)"], "nameplate capacity column")
util_col = pick_col(plnt23, ["UTLNAME", "UTILITY_NAME", "Utility name"], "utility name column")

# --- Clean + spatial filter to AOI ---
plnt23[lat_col] = pd.to_numeric(plnt23[lat_col], errors="coerce")
plnt23[lon_col] = pd.to_numeric(plnt23[lon_col], errors="coerce")
plnt23[cap_col] = pd.to_numeric(plnt23[cap_col], errors="coerce")

plnt23 = plnt23.dropna(subset=[lat_col, lon_col]).copy()

gdf_plants = gpd.GeoDataFrame(
    plnt23,
    geometry=gpd.points_from_xy(plnt23[lon_col], plnt23[lat_col]),
    crs="EPSG:4326"
)

gdf_plants_aoi = gpd.sjoin(
    gdf_plants.to_crs(gdf_aoi.crs),
    gdf_aoi[["geometry"]],
    how="inner",
    predicate="within"
).drop(columns=["index_right"], errors="ignore")

# --- Plotly map ---
plot_df = gdf_plants_aoi.copy()
plot_df["Nameplate Capacity (MW)"] = plot_df[cap_col]

fig = px.scatter_geo(
    plot_df,
    lat=lat_col,
    lon=lon_col,
    hover_name=plant_col,
    hover_data={
        plant_col: False,                  # already in hover_name
        "Nameplate Capacity (MW)": ":,.2f",
        util_col: True
    },
    title=f"eGRID PLNT23 Generation Plants in AOI (n={len(plot_df)})",
)

fig.update_geos(
    projection_type="albers usa",
    fitbounds="locations",
    visible=False
)

fig.update_traces(marker=dict(size=6, color="crimson", opacity=0.75))
fig.update_layout(height=700, width=1100, template="plotly_white")
fig.show()
